In [ ]:
library(igraph)
source("Functions.R") #include some helpers

# Lesson 2 

Network structure. In this notebook we will cover how to study the structure of networks at different biological scales:
- Local scale (species)
- Global scale (community)
- Meso-scale (groups)

Apart from the functions included in *igraph*, we will use the package *bipartite* to work with mutualistic networks, as it includes many fucntions that will make our live easier, as long as we understand what they do.

## 2A Local scale (species)

### Centrality metrics I

Let's take a look at these two networks, what is the bigger difference?

![title](./images/figure7.png)

The most basic structural properties of a network are the number of nodes (**N**) and the number of links (**L**). However, how these links are distributed among the nodes (**$K_i$**) has deep implications for other network properties (it is not the same to have all nodes with similar degree, or having a very heterogeneous dostribution). 

The degree distribution will play a very important role determining other structural metrics in the networks.
We can see the **degree distribution** of a network by doing a histogram of the degree series. This will tell us how many nodes with a given number of neighbours are in the network.

*igraph* has functions to get the degree sequence `degree(G)` (that we have already seen) and to get the degree distribution (i.e. how many nodes with a given degree are in the network) `degree_distribution(G)`.

Let's start with our foodweb:

In [ ]:
#Load the foodweb
FW_filename<- "./Data/FW_st_marks.csv"
FW_Ilist <- read.csv(FW_filename)
FW <- graph_from_data_frame(FW_Ilist,directed = TRUE)

#obtain the degree sequences
Kin=igraph::degree(FW,mode="in")
Kout=igraph::degree(FW,mode="out")

#obtain the degree distributions
Pin=degree_distribution(FW, mode="in")
Pout=degree_distribution(FW, mode="out")


op <- par(mfrow = c(1, 2), mar = c(4, 4, 3, 1))  # 1 row, 2 columns
barplot(degree_distribution(FW,mode="in"), names.arg=as.character(0:max(igraph::degree(FW,mode="in"))),  main='IN-degree of the FW')
barplot(degree_distribution(FW,mode="out"), names.arg=as.character(0:max(igraph::degree(FW,mode="out"))),  main='OUT-degree of the FW')
par(op)  # restore old graphics settings

We can obtain simple statistics like the average degree, and the standard deviation

In [ ]:
Kin_mean=mean(Kin)
Kin_sd=sd(Kin)
Kin_mean
Kin_sd

In [ ]:
Kout_mean=mean(Kout)
Kout_sd=sd(Kout)
Kout_mean
Kout_sd

There has been a lot of debate regarding the form of the degree distribution ($P(K)$) in real networks. The best practice to determine which function fits better the $P(K)$) is to use the **cumulative degree distribution** (i.e. how many nodes with degree $K$ or below are in the network) because it is less noisi. 
Let's see how we can do this.

Let's compare the degree distribution of the empirical network with a random one:

In [ ]:
n <- vcount(FW)
m <- ecount(FW)            # preserves mean degrees exactly (m/n)

G0 <- sample_gnm(n, m, directed = TRUE, loops = TRUE)

#obtain the degree sequences
Kin_rnd=igraph::degree(G0,mode="in")
Kout_rnd=igraph::degree(G0,mode="out")

#obtain the degree distributions
Pin_rnd=degree_distribution(G0, mode="in")
Pout_rnd=degree_distribution(G0, mode="out")


op <- par(mfrow = c(1, 2), mar = c(4, 4, 3, 1))  # 1 row, 2 columns
barplot(degree_distribution(G0,mode="in"), names.arg=as.character(0:max(Kin_rnd)),  main='IN-degree of the random FW')
barplot(degree_distribution(G0,mode="out"), names.arg=as.character(0:max(Kout_rnd)),  main='OUT-degree of the random FW')
par(op)  # restore old graphics settings

In [ ]:
df_FW <- cdf_get(Kout)
df_G0 <- cdf_get(Kout_rnd)

kmax <- max(df_FW$k, df_G0$k)


# 4) plot as points
plot(df_FW$k, df_FW$p,
     pch = 16,
     xlab = "k (out-degree)",
     log="xy",
     ylab = "P(K ≥ k)",
     main = "Cumulative -degree distribution",
     ylim = c(0.01, 1))

points(df_G0$k, df_G0$p, pch = 1)

legend("topright",
       legend = c("Food web (FW)", "Fake Food web (G0)"),
       pch = c(16, 1),
       bty = "n")

#### Generalizing "neighbors" to arbitrarily-sized graphs

The concept of neighbors is simple and appealing,
but it leaves us with a slight point of dissatisfaction:
it is difficult to compare graphs of different sizes.
Is a node more important solely because it has more neighbors?
What if it were situated in an extremely large graph?
Would we not expect it to have more neighbors?

As such, we need a normalization factor.
One reasonable one, in fact, is
_the number of nodes that a given node could **possibly** be connected to._
By taking the ratio of the number of neighbors a node has
to the number of neighbors it could possibly have,
we get the **degree centrality** metric.

Formally defined, the degree centrality of a node (let's call it $d$)
is the number of neighbors that a node has (let's call it $k$, its degree)
divided by the number of neighbors it could _possibly_ have (let's call it $N$, all nodes):

$$d = \frac{k}{N}$$

*igraph* does NOT provide an inbuilt function for this, so the normalization must be done by hand like this:

Degree cenrtality 

`igraph::degree(g) / (igraph::vcount(g) - 1)`

Directed versions:

`igraph::degree(g, mode="in")  / (igraph::vcount(g) - 1)`
`igraph::degree(g, mode="out") / (igraph::vcount(g) - 1)`


> Note: degree centrality is usefull to determine how "central" a node is **across** different networks, since **it does not make sense to compare the raw number of neighbours across networks with different size**!!



### Degree centraility in bipartite networks
When we are working with bipartite networks, we need to take into consideration that we have two different sets of species, and that they can only connect to species in the other set! 

Lets see some examples
In the bipartite case, the maximum possible degree of a node in a bipartite node set is the number of nodes in the opposite node set. The degree centrality for a node $u$ in the bipartite set $U$ with $n$ nodes that is connected to nodes in the bipartite set $V$ with $m$ nodes is
$d_u=\frac{k_u}{m}$, for $u\in U$, and for a node $v$ nodes in set $V$ is $d_v=\frac{k_v}{n}$, for $v\in V$, where $k_v$ is the degree of node v.

The package *bipartite* does this automatically, so we will use the functions from bipartite to work with bipartite networks

<div style="background-color:#d4edda; border-left:6px solid #28a745; padding:12px; border-radius:4px; color:#000;"><b> Up to you:</b>

<h3> Exercise 8 </h3>
Plot the cumulative degree distribution of the pollinators in the plant-pollinator network provided below, and compare with a random version of it
</div>

> Hint: you can generate a random version of the bipartite network with  
`G0 <- igraph::sample_bipartite(Na, Np, m = L, type="gnm",directed = FALSE)` 

where Na is the number of pollinators, Np number of plants, and L the number of links

In [ ]:
#your code here 
P_Filename<-"./Data/Kato_00.csv"

In [ ]:
# SOLUTION: uncomment line below to load solution
#load_and_show("./snippets/ex8.R")


The package *bipartite* has an inbuilt function that returns the fit for each guide_legend()

In [ ]:
bipartite::degreedistr(I)

Yo can retrieve just the fits withtout the figure too

In [ ]:
bipartite::networklevel(I, index="degree distribution")

### Common centrality metrics

As we have seen before, one can be "important" in different ways. 
*igraph* provides the most common metrics of node centrality
- `degree(g)`                        # Degree centrality
- `closeness(g)`                     # Closeness centrality
- `betweenness(g)`                   # Vertex betweenness centrality
- `eigen_centrality(g)$vector`       # Eigenvector centrality
- `page_rank(FW)$vector`                  # Page rank
- `V(FW)$kcore <- igraph::coreness(FW)`   # Kcore 


The importances of the nodes does not always necesarily coincides. Let's see it in our foodweb

In [ ]:
#compute several metrics of node centrality, and store them in the graph nodes
V(FW)$degree <- igraph::degree(FW)                        # Degree centrality
V(FW)$eig <- igraph::eigen_centrality(FW)$vector          # Eigenvector centrality
V(FW)$closeness <- igraph::closeness(FW)                  # Closeness centrality
V(FW)$betweenness <- igraph::betweenness(FW)              # Vertex betweenness centrality
V(FW)$pr <- igraph::page_rank(FW)$vector                  # Page rank
V(FW)$kcore <- igraph::coreness(FW)                       # Kcore 

#createa a datafram of centrlity metrics 
centrality_df <- data.frame(row.names   = V(FW)$name,
                         degree      = V(FW)$degree,
                         closeness   = V(FW)$closeness,
                         betweenness = V(FW)$betweenness,
                         eigenvector = V(FW)$eig,
                         pr = V(FW)$pr,
                         kcore= V(FW)$kcore
                         )


It is possible to color the nodes according to a given property, like this

In [ ]:
metric <- V(FW)$pr   # <-- choose metric here betweenness, closeness, eig, pr
cols <- metric_to_color(metric)
plot_as_flux(FW, vertex.color = cols)

### K_core decomposition
The k-core of a graph is a maximal subgraph in which every vertex has at least degree kk.
It’s a way to iteratively remove nodes with degree less than kk, resulting in progressively smaller subgraphs.
k-core decomposition identifies these subgraphs for varying values of kk. The larger the kk, the more "central" or "core" the remaining nodes are considered to be in the graph's structure.

For example, a 2-core would be a subgraph where all nodes have at least degree 2, meaning each node is connected to at least two other nodes.
The process continues by removing nodes with degrees less than kk until the condition is met.

### Page Rank

PageRank computes a ranking of the nodes in the graph G based on the structure of the incoming links. It was originally designed as an algorithm to rank web pages.
However, it can also be used to identify the species that "move" more biomass trough a network, or in general, the node that is most used when trasnporting information trough the graph. Since this is only interesting in **directed graphs** let's use one of our directed networks. It has been used, for example, to find what are the nodes that are pointing to the more "important" nodes, in order to find the species that can cause more harm when they disapear from the network.

<div style="background-color:#d4edda; border-left:6px solid #28a745; padding:12px; border-radius:4px; color:#000;"><b> Up to you:</b>

<h3> Exercise 9 </h3>
Explore the centrality of the nodes according tot the different metrics. 
¿Are they similar? 

- What node would you target if you want to prevent a disease spreading trough the network?
- What node would you target if you want tear down the foodweb as soon as possible?
- Cuantify the correlation between the different centrality metrics

</div>

In [ ]:
#your code here

In [ ]:
# SOLUTION: uncomment line below to load solution
#load_and_show("./snippets/ex9.R")

### Common centrality metrics in bipartite networks

While package bipartite has many functions to quantify cnetrality, most of them (all that use an interaction list as variable) are designed for one mode networks, so it is easier to keep working in *igraph* for this.
The only one that is relevant are the normalized degrees: generality/vulnerability

In [ ]:
V(B)$degree <- igraph::degree(B)                        # Degree centrality
V(B)$eig <- igraph::eigen_centrality(B)$vector                    # Eigenvector centrality
V(B)$closeness <- igraph::closeness(B)                  # Closeness centrality
V(B)$betweenness <- igraph::betweenness(B)              # Vertex betweenness centrality
V(B)$pr <- igraph::page_rank(B)$vector                  # Page rank
V(B)$kcore <- igraph::coreness(B)                       # Kcore 

res <- get_MusRank(I, mode = "ranking", niterations = 100, seed = 1)
V(B)[!type]$FC <- res$animal_fitness[V(B)[!type]$name]  #fitness-complexity for animals and plants
V(B)[ type]$FC <- res$plant_complexity[V(B)[ type]$name]

centrality_df <- data.frame(row.names   = V(B)$name,
                         type = V(B)$type,
                         degree      = V(B)$degree,
                         closeness   = V(B)$closeness,
                         betweenness = V(B)$betweenness,
                         eigenvector = V(B)$eig,
                         pagerank = V(B)$pr,
                         core = V(B)$kcore,
                         FC = V(B)$FC)

In [ ]:
metric <- V(B)$FC 
metric

In [ ]:
metric <- V(B)$kcore  # <-- choose metric here betweenness, closeness, eig, pr  
#layout_with_fr()        # Fruchterman–Reingold (classic default)
#layout_with_kk()        # Kamada–Kawai (distance-preserving)
#layout_with_drl() 
cols <- metric_to_color(metric)
plot(B, vertex.color = cols,vertex_size=0.5,vertex.label=NA)

In [ ]:
cor(centrality_df, method = "spearman") 